In [ ]:

from soundmining_library.piece import Piece

piece = Piece()
await piece.start(should_send_to_score=False)

In [ ]:
%%html
<style>
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
.jp-OutputArea-output {
   background-color: transparent;
}  
</style>

In [ ]:
import random
import time
from dataclasses import dataclass
from enum import StrEnum
from itertools import permutations

import ipywidgets as widgets
from IPython.display import display
from soundmining_library.generative import MarkovChain, pan_point, random_int_range, random_range
from soundmining_library.modular.instrument import AddAction, AudioInstrument
from soundmining_library.modular_v2.synth_player_v2 import SynthPlayerV2
from soundmining_library.sequencer import SequenceNote
from soundmining_library.spectrum import make_fact, make_spectrum, make_undertone_spectrum
from soundmining_library.supercollider_client import SupercolliderClient
from soundmining_library.supercollider_receiver import ExtendedNoteHandler, PatchArguments
from soundmining_library.ui.ui_controls import UiControls
from soundmining_library.ui.ui_piece_model import UiPieceBuilder


static_control = piece.instruments.static_control
sine_control = piece.instruments.sine_control
line_control = piece.instruments.line_control
two_block_sine_control = piece.instruments.two_block_sine_control
impulse_osc = piece.instruments.impulse_osc
dust_osc = piece.instruments.dust_osc
signal_sum = piece.instruments.signal_sum
signal_multiply = piece.instruments.signal_multiply
lf_noise_osc = piece.instruments.lf_noise_osc

SOUNDPATH = piece.environment.sound_path

Sounds = StrEnum("Sounds", ["H1", "H2", "K1", "Ka1", "Ki1", "Ko1", "S1", "S2"])


@dataclass
class SoundData:
    sound: Sounds
    file_name: str
    duration: float
    fundamental: float
    first_partial: float
    start_time: float
    end_time: float
    peak_time: float

    def make_fact(self) -> float:
        return make_fact(fundamental=self.fundamental, first_partial=self.first_partial)

    def make_spectrum(self) -> list[float]:
        fact = self.make_fact()
        return make_spectrum(fundamental=self.fundamental, fact=fact, size=SPECTRUM_SIZE)

    def make_undertone_spectrum(self) -> list[float]:
        fact = self.make_fact()
        return make_undertone_spectrum(fundamental=self.fundamental, fact=fact, size=SPECTRUM_SIZE)

    def get_relative_peak_time(self, reverse: bool = False) -> tuple[float, float]:
        relative_peak_time = (self.peak_time - self.start_time) / (self.end_time - self.start_time)
        if reverse:
            relative_peak_time = 1.0 - relative_peak_time
        relative_times = (relative_peak_time, 1.0 - relative_peak_time)

        return relative_times

    def get_relative_start_end(self) -> tuple[float, float]:
        return (self.start_time / self.duration, self.end_time / self.duration)

    def make_rate(self, second_freq: float) -> float:
        return second_freq / self.fundamental

    def add_sound_to_synth_player(self, synth_player: SynthPlayerV2):
        synth_player.add_sound(self.sound, self.file_name, self.start_time, self.end_time)

    def get_play_duration(self, rate: float) -> float:
        return (self.end_time - self.start_time) * rate


all_sound_data = {
    Sounds.H1: SoundData(
        sound=Sounds.H1,
        file_name=f"{SOUNDPATH}/H1.aif",
        duration=1.781,
        fundamental=959,
        first_partial=1565,
        start_time=0.234,
        end_time=1.395,
        peak_time=0.410,
    ),
    Sounds.H2: SoundData(
        sound=Sounds.H2,
        file_name=f"{SOUNDPATH}/H2.aif",
        duration=1.414,
        fundamental=933,
        first_partial=1307,
        start_time=0.105,
        end_time=1.216,
        peak_time=0.281,
    ),
    Sounds.K1: SoundData(
        sound=Sounds.K1,
        file_name=f"{SOUNDPATH}/K1.aif",
        duration=0.896,
        fundamental=490,
        first_partial=1696,
        start_time=0.196,
        end_time=0.779,
        peak_time=0.233,
    ),
    Sounds.Ka1: SoundData(
        sound=Sounds.Ka1,
        file_name=f"{SOUNDPATH}/Ka1.aif",
        duration=0.757,
        fundamental=1350,
        first_partial=1678,
        start_time=0.249,
        end_time=0.714,
        peak_time=0.315,
    ),
    Sounds.Ki1: SoundData(
        sound=Sounds.Ki1,
        file_name=f"{SOUNDPATH}/Ki1.aif",
        duration=0.649,
        fundamental=492,
        first_partial=636,
        start_time=0.164,
        end_time=0.570,
        peak_time=0.211,
    ),
    Sounds.Ko1: SoundData(
        sound=Sounds.Ko1,
        file_name=f"{SOUNDPATH}/Ko1.aif",
        duration=0.721,
        fundamental=567,
        first_partial=1080,
        start_time=0.169,
        end_time=0.630,
        peak_time=0.220,
    ),
    Sounds.S1: SoundData(
        sound=Sounds.S1,
        file_name=f"{SOUNDPATH}/S1.aif",
        duration=2.110,
        fundamental=521,
        first_partial=639,
        start_time=0.170,
        end_time=1.834,
        peak_time=0.584
    ),
    Sounds.S2: SoundData(
        sound=Sounds.S2,
        file_name=f"{SOUNDPATH}/S2.aif",
        duration=1.749,
        fundamental=466,
        first_partial=1023,
        start_time=0.192,
        end_time=1.693,
        peak_time=0.690
    )
}


def _get_variable_start_end(the_arg: float | tuple[float, float] | None) -> tuple[float, float]:
        match the_arg:
            case (start, end):
                start = start
                end = end
            case float(val) | int(val):
                start = val
                end = val

        return (start, end)

def make_grain_trigger_bus(
    impulse_freq: float | tuple[float, float] | None, dust_freq: float | tuple[float, float] | None = None
) -> AudioInstrument:
    impulse_start, impulse_end = _get_variable_start_end(impulse_freq)
    if impulse_start:
        impulse_trigger = impulse_osc(
            amp_bus=static_control(1.0), freq_bus=line_control(impulse_start, impulse_end)
        ).add_action(AddAction.TAIL_ACTION)

    dust_start, dust_end = _get_variable_start_end(dust_freq)
    if dust_start:
        dust_trigger = dust_osc(
            amp_bus=static_control(1.0), freq_bus=line_control(dust_start, dust_end)
        ).add_action(AddAction.TAIL_ACTION)

    match (impulse_trigger, dust_trigger):
        case (imp, dust):
            return signal_sum(imp, dust).add_action(AddAction.TAIL_ACTION)
        case (imp, None) if imp is not None:
            return imp
        case (None, dust) if dust is not None:
            return dust

def make_grain_duration_bus(        
    grain_duration: float | tuple[float, float],
    grain_duration_noise: float | tuple[float, float] | None = None,
    lf_noise_rate: float = 500,
) -> AudioInstrument:
    grain_duration_start, grain_duration_end = _get_variable_start_end(grain_duration)
    grain_duration_line = line_control(grain_duration_start, grain_duration_end).add_action(AddAction.TAIL_ACTION)
    match grain_duration_noise:
        case (noise_lower, noise_upper):
            grain_duration_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_lower,
                high_value=noise_upper,
            )
        case float(noise_val) | int(noise_val):
            grain_duration_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_val if noise_val < 1.0 else 1.0,
                high_value=noise_val if noise_val > 1.0 else 1.0,
            )
        case None:
            grain_duration_lf_noise = None

    if grain_duration_lf_noise is not None:
        return signal_multiply(grain_duration_line, grain_duration_lf_noise).add_action(AddAction.TAIL_ACTION)
    else:
        return grain_duration_line

def make_grain_rate_bus(    
    grain_rate: float | tuple[float, float],
    grain_rate_noise: float | tuple[float, float] | None = None,
    lf_noise_rate: float = 500,
) -> AudioInstrument:
    grain_rate_start, grain_rate_end = _get_variable_start_end(grain_rate)
    grain_rate_line = line_control(grain_rate_start, grain_rate_end).add_action(AddAction.TAIL_ACTION)
    match grain_rate_noise:
        case (noise_lower, noise_upper):
            grain_rate_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_lower,
                high_value=noise_upper,
            )
        case float(noise_val) | int(noise_val):
            grain_rate_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_val if noise_val < 1.0 else 1.0,
                high_value=noise_val if noise_val > 1.0 else 1.0,
            )
        case None:
            grain_rate_lf_noise = None

    if grain_rate_lf_noise is not None:
        return signal_multiply(grain_rate_line, grain_rate_lf_noise).add_action(AddAction.TAIL_ACTION)
    else:
        return grain_rate_line

def make_grain_pos_bus(    
    sound_data: SoundData,
    reversed: bool,
    grain_pos_noise: float | tuple[float, float] | None = None,
    lf_noise_rate: float = 500,
) -> AudioInstrument:
    relative_start, relative_end = sound_data.get_relative_start_end()

    if reversed:
        pos_end = relative_start
        pos_start = relative_end
    else:
        pos_start = relative_start
        pos_end = relative_end

    line_pos = line_control(pos_start, pos_end).add_action(AddAction.TAIL_ACTION)

    match grain_pos_noise:
        case (noise_lower, noise_upper):
            grain_pos_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_lower,
                high_value=noise_upper,
            )
        case float(noise_val) | int(noise_val):
            grain_pos_lf_noise = lf_noise_osc(
                amp_bus=static_control(1.0),
                freq_bus=static_control(lf_noise_rate),
                low_value=noise_val * -1.0,
                high_value=noise_val,
            )
        case None:
            grain_pos_lf_noise = None

    if grain_pos_lf_noise is not None:
        return signal_sum(line_pos, grain_pos_lf_noise).add_action(AddAction.TAIL_ACTION)
    else:
        return line_pos
        
def play_sound_with_grainbuf(    
    start: float,
    sound_data: SoundData,
    grain_pos_bus: AudioInstrument,        
    pan_position: AudioInstrument,
    grain_trigger_bus: AudioInstrument,
    grain_duration_bus: AudioInstrument,
    grain_rate_bus: AudioInstrument,
    volume: float,
    play_rate: float,        
):
    sound = sound_data.sound
    
    duration = sound_data.get_play_duration(play_rate)
    (
        piece.synth_player.note()
        .mono_grain_buf(sound, grain_trigger_bus, grain_duration_bus, grain_rate_bus, grain_pos_bus)
        .mono_volume(sine_control(0, volume))
        .pan(pan_position)
        .play(start_time=start, duration=duration)
    )


piece.reset()

for sound_data in all_sound_data.values():
    sound_data.add_sound_to_synth_player(piece.synth_player)

piece.synth_player.start()


ui_controls = (
    UiControls(piece)
    .header_label("Live Controls")
    .range_floatslider("pan", "Pan range")
    .divider()
    .header_label("Piece Canvas")
    .piece_canvas()
    .header_label("Sounds")
    .sound_grid()
    .stop_button()
    .render())

sequence_notes = [
    SequenceNote(0, "Track 1", duration=3, note=1, peak=0.5),
    SequenceNote(1, "Track 1", duration=3, note=2, peak=0.5, color="Red"),
    SequenceNote(3, "Track 1", duration=5, note=3, peak=0.5),
    SequenceNote(0, "Track 2", duration=3, note=2, peak=0.5),
    SequenceNote(2, "Track 2", duration=5, note=2, peak=0.33, color="Blue"),
    SequenceNote(8, "Track 3", duration=5, note=1, peak=0.33),
]

ui_piece = UiPieceBuilder().add_notes(sequence_notes).build()
ui_controls.draw_piece(ui_piece)

class MyHandler(ExtendedNoteHandler):
    def __init__(self, client: SupercolliderClient, pan_widget: widgets.FloatRangeSlider):
        super().__init__(client)
        self._pan_widget = pan_widget

    def handle_note(self, patch_arguments: PatchArguments):    
        low, high = self._pan_widget.value                
        (
            piece.synth_player.note()
            .sine(freq=static_control(patch_arguments.pitch), amp=sine_control(0, patch_arguments.amp))
            .pan(static_control(random_range(low, high)))
            .play(patch_arguments.start, 3)
         )
        (
            piece.synth_player.note()
            .sine(freq=static_control(patch_arguments.pitch * 2), amp=sine_control(0, patch_arguments.amp))
            .pan(static_control(random_range(low, high)))
            .play(patch_arguments.start + 3, 3)
         )

pan_range_slider = ui_controls.float_range_sliders["pan"]
my_handler = MyHandler(piece.supercollider_client, pan_range_slider)
piece.receiver.set_note_handler(my_handler)

In [ ]:
piece.stop()